In [1]:
"""異쒗빆 �쇱젙 �댄깉(�뺤떆��) 媛쒖꽑�� 洹쇨굅 洹몃옒�� (2019.01~2024.12).

�곗씠��: �낆텧��_吏묎퀎�뺣낫.csv �� �뚯씪留� �ъ슜. ���곴낵 �댁긽媛� �쒖쇅 湲곗��� cause_charts.py�� 媛숇떎.
異쒗빆 吏��� = 異쒗빆�쇱떆 - 異쒗빆�덉젙�쇱떆. 6�쒓컙 �섍쾶 ��쑝硫� '吏���'�쇰줈 蹂몃떎.

�ъ슜踰�
    python dep_delay_charts.py �낆텧��_吏묎퀎�뺣낫.csv [--out 寃곌낵�대뜑]
"""
import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cause_charts import BLUE, ORANGE, GRAY, INK2, SURFACE, YEARS, load, save, style_year_axis

NOTE = "�먮즺: 遺��고빆留뚭났�� �좊컯 �낆텧�� 吏묎퀎 �뺣낫, 而⑦뀒�대꼫��(��쨌�몃�) �낇빆 湲곗�"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv", type=Path)
    ap.add_argument("--out", type=Path, default=Path("charts_異쒗빆吏���"))
    args = ap.parse_args()
    args.out.mkdir(parents=True, exist_ok=True)

    c = load(args.csv)
    d = c[c.dep_delay.between(-72, 240)].copy()      # �덉젙�쇱떆 �낅젰 �ㅻ쪟(3200�� ��) �쒖쇅
    d["late"] = d.dep_delay > 6
    d["big"] = d["珥앺넠��"] >= 100000

    # �� 1: �곕룄蹂� 吏��� 吏���
    g = d.groupby("y")
    t = pd.DataFrame({
        "嫄댁닔": g.size(),
        "6h+吏���_%": g.late.mean() * 100,
        "12h+吏���_%": g.dep_delay.apply(lambda s: (s > 12).mean() * 100),
        "24h+吏���_%": g.dep_delay.apply(lambda s: (s > 24).mean() * 100),
        "10留똆T+_6h+吏���_%": d[d.big].groupby("y").late.mean() * 100,
        "10留똆T誘몃쭔_6h+吏���_%": d[~d.big].groupby("y").late.mean() * 100,
        "吏��곗꽑諛�_�ы빆�쒓컙_以묒븰媛�_h": d[d.late].groupby("y").stay.median(),
        "�뺤떆�좊컯_�ы빆�쒓컙_以묒븰媛�_h": d[~d.late].groupby("y").stay.median(),
        "�덉젙_�ы빆�쒓컙_以묒븰媛�_h": ((d.etd - d.ata).dt.total_seconds() / 3600).groupby(d.y).median(),
        "�덉젙=�ㅼ젣_�뺥솗�쇱튂_%": g.dep_delay.apply(lambda s: (s == 0).mean() * 100),
    }).round(1)
    t.to_csv(args.out / "異쒗빆吏���_�곕룄蹂�.csv", encoding="utf-8-sig")
    print(t.T.to_string())

    # 洹몃옒�� A: �좊컯 �ш린蹂� 吏��� 鍮꾩쨷
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for col, color, label in [("10留똆T+_6h+吏���_%", ORANGE, "10留똆T �댁긽"),
                              ("10留똆T誘몃쭔_6h+吏���_%", BLUE, "10留똆T 誘몃쭔")]:
        ax.plot(t.index, t[col], color=color, marker="o", markersize=7,
                markeredgecolor=SURFACE, markeredgewidth=1.5, label=label)
        for x, v in t[col].items():
            ax.annotate(f"{v:.1f}", (x, v), textcoords="offset points", xytext=(0, 8),
                        ha="center", fontsize=9.5, color=color)
    style_year_axis(ax)
    ax.set_ylim(0, 25)
    ax.set_ylabel("異쒗빆 �덉젙蹂대떎 6�쒓컙 �섍쾶 ��� 鍮꾩쨷(%)")
    ax.set_title("���뺤꽑�� 異쒗빆 吏��곗� 2019�꾩쓽 3諛� �섏��먯꽌 �대젮�ㅼ� �딆븯��", loc="left", fontsize=13)
    ax.legend(frameon=False, loc="upper left")
    fig.text(0.01, 0.01, NOTE, fontsize=8.5, color=INK2)
    save(fig, args.out, "A_�ш린蹂�_異쒗빆吏���.png")

    # 洹몃옒�� B: �꾩갑�� ��뿀�� �뺣룄蹂� 異쒗빆 吏��� 鍮꾩쨷 (吏��곗씠 諛뽰뿉�� �ㅻ뒗吏�, ��쭔 �덉뿉�� �앷린�붿�)
    v = d[d.voy.between(0, 720)].copy()
    v["excess"] = v.voy - v.groupby("�꾩텧���肄붾뱶2").voy.transform("median")
    v["arr"] = pd.cut(v.excess, [-1e9, 0, 6, 12, 24, 1e9],
                      labels=["�됱냼蹂대떎 鍮좊쫫", "0~6h ��쓬", "6~12h ��쓬", "12~24h ��쓬", "24h+ ��쓬"])
    b = v.groupby("arr", observed=True).late.mean() * 100
    b.round(1).to_csv(args.out / "�꾩갑吏��곕퀎_異쒗빆吏���.csv", encoding="utf-8-sig")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(range(len(b)), b.values, width=0.55, color=BLUE)
    for i, val in enumerate(b.values):
        ax.annotate(f"{val:.1f}%", (i, val), textcoords="offset points", xytext=(0, 4),
                    ha="center", fontsize=10, color=INK2)
    ax.set_xticks(range(len(b)), b.index)
    ax.set_ylim(0, 20)
    ax.set_xlabel("吏곸쟾 ��뎄�먯꽌 遺��곌퉴吏� 嫄몃┛ �쒓컙 (媛숈� 異쒕컻�� 以묒븰媛� ��鍮�)")
    ax.set_ylabel("異쒗빆 6�쒓컙+ 吏��� 鍮꾩쨷(%)")
    ax.set_title("��쾶 �꾩갑�� 諛곕룄 異쒗빆 吏��� 鍮꾩쨷�� �ш쾶 �ㅻⅤ吏� �딅떎", loc="left", fontsize=13)
    fig.text(0.01, 0.01, NOTE + ", 2019~2024 �⑷퀎", fontsize=8.5, color=INK2)
    save(fig, args.out, "B_�꾩갑吏��곕퀎_異쒗빆吏���.png")

    # 洹몃옒�� C: 諛섎났 吏��� �좊컯 吏묒쨷�� (2023~2024, 5�� �댁긽 �낇빆�� �좊컯)
    r = d[d.y >= 2023].groupby("�좊컯��").agg(n=("late", "size"), late=("late", "sum"))
    r = r[r.n >= 5]
    rep = (r.late / r.n) >= 0.3
    share = pd.DataFrame({
        "諛섎났 吏��� �좊컯 (吏��곗쑉 30%+)": [rep.mean() * 100, r[rep].n.sum() / r.n.sum() * 100,
                                  r[rep].late.sum() / r.late.sum() * 100],
        "�섎㉧吏� �좊컯": [(~rep).mean() * 100, r[~rep].n.sum() / r.n.sum() * 100,
                   r[~rep].late.sum() / r.late.sum() * 100],
    }, index=["�좊컯 ��", "�낇빆 嫄댁닔", "6h+ 吏��� 嫄댁닔"]).round(1)
    share.to_csv(args.out / "諛섎났吏��곗꽑諛�_吏묒쨷��.csv", encoding="utf-8-sig")
    print(f"\n諛섎났 吏��� �좊컯 {rep.sum()}泥� / �꾩껜 {len(r)}泥�")
    print(share.to_string())
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = range(len(share))
    ax.bar(x, share.iloc[:, 0], width=0.55, color=ORANGE, label=share.columns[0])
    ax.bar(x, share.iloc[:, 1], width=0.55, bottom=share.iloc[:, 0], color=GRAY, label=share.columns[1])
    for i, val in enumerate(share.iloc[:, 0]):
        ax.annotate(f"{val:.1f}%", (i, val / 2), ha="center", va="center", fontsize=11,
                    color="white", fontweight="bold")
    ax.set_xticks(list(x), share.index)
    ax.set_ylim(0, 100)
    ax.set_ylabel("鍮꾩쨷(%)")
    ax.set_title(f"�좊컯 {rep.mean()*100:.0f}%媛� 異쒗빆 吏��곗쓽 {share.iloc[2, 0]:.0f}%瑜� �몃떎 (2023~2024)",
                 loc="left", fontsize=13)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
    fig.text(0.01, 0.01, NOTE + ", 5�� �댁긽 �낇빆 �좊컯", fontsize=8.5, color=INK2)
    save(fig, args.out, "C_諛섎났吏��곗꽑諛�_吏묒쨷��.png")

    # ---- 愿�由� 諛⑸쾿蹂� 洹쇨굅 ----
    # ��쭔 �� 而⑦뀒�대꼫�� ��(�낇빆 �쒖젏): 洹몃븣源뚯� �낇빆�� �� - 洹몃븣源뚯� 異쒗빆�� ��
    A, D = np.sort(c.ata.values), np.sort(c.atd.values)
    d["in_port"] = np.searchsorted(A, d.ata.values, side="left") - np.searchsorted(D, d.ata.values, side="right")
    d["pstay"] = (d.etd - d.ata).dt.total_seconds() / 3600
    d = d[d.pstay > 0]
    # 臾쇰웾 湲곗� �꾩슂 �묒뾽�쒓컙 = 泥섎━ �붾Ъ / 2019�� 媛숈� �ш린 �좊컯�� 泥섎━�띾룄 以묒븰媛�(t/h)
    d["sz"] = pd.cut(d["珥앺넠��"], [0, 10000, 30000, 60000, 100000, 1e7])
    d["t_per_h"] = d.work / d.stay
    ref = d[d.y == 2019].groupby("sz", observed=True).t_per_h.median()
    d["gap"] = d.pstay - d.work / d.sz.map(ref).astype(float)
    labels = ["12h+ 遺�議�", "6~12h 遺�議�", "�곸젙(짹6h)", "6~12h �ъ쑀", "12h+ �ъ쑀"]
    d["gap_b"] = pd.cut(d.gap, [-1e9, -12, -6, 6, 12, 1e9], labels=labels)
    gp = d.pivot_table(index="gap_b", columns="y", values="late", aggfunc="mean", observed=True) * 100
    gp.round(1).to_csv(args.out / "怨꾪쉷遺�議깅퀎_異쒗빆吏���.csv", encoding="utf-8-sig")
    print("\n怨꾪쉷 �ы빆�쒓컙 遺�議� �뺣룄蹂� 6h+ 吏��� 鍮꾩쨷(%)\n", gp.round(1).to_string())

    fig, ax = plt.subplots(figsize=(8, 4.5))
    vals = gp[2024]
    colors = [ORANGE if l.endswith("遺�議�") else BLUE for l in vals.index]
    ax.bar(range(len(vals)), vals.values, width=0.55, color=colors)
    for i, val in enumerate(vals.values):
        ax.annotate(f"{val:.1f}%", (i, val), textcoords="offset points", xytext=(0, 4),
                    ha="center", fontsize=10, color=INK2)
    ax.set_xticks(range(len(vals)), vals.index)
    ax.set_ylim(0, 20)
    ax.set_xlabel("異쒗빆 �덉젙源뚯� �쒓컙 - 臾쇰웾 湲곗� �꾩슂 �묒뾽�쒓컙")
    ax.set_ylabel("異쒗빆 6�쒓컙+ 吏��� 鍮꾩쨷(%)")
    ax.set_title("�묒뾽�쒓컙�� 吏㏐쾶 �≪� 諛곌� �� ��쾶 �좊궃�� (2024)", loc="left", fontsize=13)
    fig.text(0.01, 0.01, NOTE + ", 泥섎━�띾룄 湲곗��� 2019�� �ш린蹂� 以묒븰媛�", fontsize=8.5, color=INK2)
    save(fig, args.out, "D_怨꾪쉷遺�議깅퀎_異쒗빆吏���.png")

    # ���뺤꽑 泥섎━�띾룄(�ы빆 1�쒓컙�� 泥섎━ �붾Ъ)
    sp = d[d.big].groupby("y").t_per_h.median()
    sp.round(0).to_csv(args.out / "���뺤꽑_泥섎━�띾룄.csv", encoding="utf-8-sig")
    print("\n10留똆T �댁긽 泥섎━�띾룄 以묒븰媛�(t/h)\n", sp.round(0).to_string())
    fig, ax = plt.subplots(figsize=(8, 4.5))
    from cause_charts import bars_by_year
    bars_by_year(ax, sp, "{:,.0f}")
    ax.set_ylabel("�ы빆 1�쒓컙�� 泥섎━ �붾Ъ(��)")
    ax.set_title("10留똆T �댁긽 �좊컯�� 泥섎━�띾룄�� 2019�꾨낫�� 17% �먮━��", loc="left", fontsize=13)
    fig.text(0.01, 0.01, NOTE + ", (�곸옱+�묓븯 �ㅼ닔) / �ы빆�쒓컙", fontsize=8.5, color=INK2)
    save(fig, args.out, "E_���뺤꽑_泥섎━�띾룄.png")

    # ��쭔 �� �좊컯 �섎퀎 吏��� 鍮꾩쨷
    d["in_port_b"] = pd.cut(d.in_port, [-1, 32, 35, 38, 42, 200],
                            labels=["32泥� �댄븯", "33~35泥�", "36~38泥�", "39~42泥�", "43泥� �댁긽"])
    cp = d.pivot_table(index="in_port_b", columns="y", values="late", aggfunc="mean", observed=True) * 100
    cp.round(1).to_csv(args.out / "��궡�좊컯�섎퀎_異쒗빆吏���.csv", encoding="utf-8-sig")
    print("\n�낇빆 �쒖젏 ��쭔 �� 而⑦뀒�대꼫�� �섎퀎 6h+ 吏��� 鍮꾩쨷(%)\n", cp.round(1).to_string())
    fig, ax = plt.subplots(figsize=(8, 4.5))
    allv = d.groupby("in_port_b", observed=True).late.mean() * 100
    ax.bar(range(len(allv)), allv.values, width=0.55,
           color=[ORANGE if i == len(allv) - 1 else BLUE for i in range(len(allv))])
    for i, val in enumerate(allv.values):
        ax.annotate(f"{val:.1f}%", (i, val), textcoords="offset points", xytext=(0, 4),
                    ha="center", fontsize=10, color=INK2)
    ax.set_xticks(range(len(allv)), allv.index)
    ax.set_ylim(0, 20)
    ax.set_xlabel("�낇빆�� �� �대� ��쭔 �덉뿉 �덈뜕 而⑦뀒�대꼫�� ��")
    ax.set_ylabel("異쒗빆 6�쒓컙+ 吏��� 鍮꾩쨷(%)")
    ax.set_title("��쭔�� 遺먮퉴 �� �ㅼ뼱�� 諛곌� �� ��쾶 �좊궃��", loc="left", fontsize=13)
    fig.text(0.01, 0.01, NOTE + ", 2019~2024 �⑷퀎", fontsize=8.5, color=INK2)
    save(fig, args.out, "F_��궡�좊컯�섎퀎_異쒗빆吏���.png")
    print(f"\n寃곌낵 ����: {args.out.resolve()}")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'matplotlib'